# Kaapi Bricks — India Workspace Setup

**Workspace:** `fevm-fe-india-gcc.cloud.databricks.com`
**Catalog:** `fe_india_gcc_catalog`
**Schema:** `kaapi_bricks`

Run all cells in order. This sets up everything needed for the Kaapi Bricks demo.

### What this notebook does:
1. Install dependencies
2. Generate 14 Delta tables (200K orders, 37 stores, 28 products)
3. Generate 6 KA PDF documents
4. Create Knowledge Assistant
5. Create Genie Space
6. Create MAS Supervisor (KA + Genie)
7. Print configuration for app deployment

In [ ]:
%pip install faker holidays numpy pandas pyarrow fpdf2 databricks-sdk mlflow[databricks]>=3.1
dbutils.library.restartPython()

In [ ]:
# Configuration — UPDATE THESE FOR YOUR WORKSPACE
CATALOG = "fe_india_gcc_catalog"
SCHEMA = "kaapi_bricks"
VOLUME_PATH = f"/Volumes/{CATALOG}/{SCHEMA}/raw_data"
WAREHOUSE_ID = "13a6b534c35ffbcb"  # Serverless Starter Warehouse

print(f"Catalog: {CATALOG}")
print(f"Schema: {SCHEMA}")
print(f"Volume: {VOLUME_PATH}")

## Step 1: Generate Data Tables
Runs the data generation script. Creates 14 Delta tables with ~200K orders.

In [ ]:
# Copy and run generate_data.py with India workspace config
# The script is at /Workspace/Users/kunal.gaurav@databricks.com/AI/Evaluation/generate_data.py
# OR paste it from the Aidays project

# For now, run it from the synced location:
%run /Workspace/Users/kunal.gaurav@databricks.com/kaapi-bricks-setup/generate_data

## Step 2: Generate KA Documents (6 PDFs)

In [ ]:
%run /Workspace/Users/kunal.gaurav@databricks.com/kaapi-bricks-setup/generate_ka_documents

## Step 3: Create Knowledge Assistant

In [ ]:
# Create KA using manage_ka MCP tool (run from Agent Bricks UI or Claude Code)
# OR create manually:
print("""
=== CREATE KNOWLEDGE ASSISTANT ===
Go to: Databricks UI > Machine Learning > Agent Bricks > Create New > Knowledge Assistant

Name: Kaapi Bricks Operations Knowledge
Volume Path: /Volumes/fe_india_gcc_catalog/kaapi_bricks/raw_data/ka_documents
Description: Answers questions about Kaapi Bricks recipes, barista training, food safety policies, equipment maintenance, franchise operations, and supplier agreements.
Instructions: Be helpful and always cite the specific document when answering. If unsure, say so. Use INR for all prices.

After creation, note the:
- KA tile_id
- KA endpoint name (e.g., ka-xxxxxxxx-endpoint)
""")

## Step 4: Create Genie Space

In [ ]:
print("""
=== CREATE GENIE SPACE ===
Go to: Databricks UI > SQL > Genie Spaces > Create New

Name: Kaapi Bricks Analytics
SQL Warehouse: Serverless Starter Warehouse
Tables to add (all from fe_india_gcc_catalog.kaapi_bricks):
  - stores
  - products
  - orders
  - order_items
  - customers
  - inventory_transactions
  - suppliers
  - ingredients
  - promotions
  - promotion_redemptions
  - purchase_orders
  - toppings
  - order_item_toppings

Sample questions to add:
  - What are the top 5 selling drinks this month?
  - Show me revenue for the Koramangala store
  - Which ingredients are running low in inventory?
  - How did the Morning Rush Hour promotion perform?

After creation, note the:
- Genie Space ID
""")

## Step 5: Create MAS Supervisor

In [ ]:
print("""
=== CREATE MAS SUPERVISOR ===
Go to: Databricks UI > Machine Learning > Agent Bricks > Create New > Supervisor Agent

Name: Kaapi Bricks HQ

Agents:
1. Knowledge Assistant:
   - Name: knowledge_assistant
   - Type: KA (select the one created in Step 3)
   - Description: Answers questions about recipes, barista training, food safety policies, equipment maintenance, franchise operations, and supplier agreements from Kaapi Bricks internal documents

2. Genie Space:
   - Name: analytics
   - Type: Genie Space (select the one created in Step 4)
   - Description: Answers data questions about sales, revenue, inventory, customers, orders, promotions, and suppliers using SQL on 14 Delta tables

Instructions: Route questions as follows: recipes/policies/training → knowledge_assistant. Sales/inventory/revenue data → analytics.

After creation, note the:
- MAS endpoint name (e.g., mas-xxxxxxxx-endpoint)
""")

## Step 6: Create po_line_items table (for invoice demo)

In [ ]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {CATALOG}.{SCHEMA}.po_line_items (
  line_id STRING, po_id STRING, ingredient_id STRING, ingredient_name STRING,
  quantity DOUBLE, unit STRING, unit_price DOUBLE, line_total DOUBLE
)
""")

# Get a recent PO for Koramangala
pos = spark.sql(f"""
  SELECT p.po_id, p.supplier_id, s.name as supplier_name
  FROM {CATALOG}.{SCHEMA}.purchase_orders p
  JOIN {CATALOG}.{SCHEMA}.suppliers s ON p.supplier_id = s.supplier_id
  JOIN {CATALOG}.{SCHEMA}.stores st ON p.store_id = st.store_id
  WHERE st.name = 'Kaapi Bricks Koramangala' AND p.supplier_id = 'SUP-001'
  ORDER BY p.order_date DESC LIMIT 1
""").collect()

if pos:
    po_id = pos[0]['po_id']
    print(f"Using PO: {po_id}")
    spark.sql(f"""
      INSERT INTO {CATALOG}.{SCHEMA}.po_line_items VALUES
      ('pl-001', '{po_id}', 'ING-001', 'Coorg Arabica Beans', 25.0, 'kg', 1200.0, 30000.0),
      ('pl-002', '{po_id}', 'ING-004', 'Chicory', 15.0, 'kg', 400.0, 6000.0),
      ('pl-003', '{po_id}', 'ING-002', 'House Blend Pre-Mix', 25.0, 'kg', 820.0, 20500.0),
      ('pl-004', '{po_id}', 'ING-021', 'Paper Cups (200ml)', 10.0, 'case', 800.0, 8000.0),
      ('pl-005', '{po_id}', 'ING-022', 'Stirrer Sticks', 5.0, 'case', 300.0, 1500.0)
    """)
    print(f"Inserted 5 PO line items for {po_id}")
else:
    print("No Coorg Coffee PO found for Koramangala. Generate data first (Step 1).")

## Step 7: Print App Configuration
After completing Steps 3-5, fill in the values below and update app.yaml

In [ ]:
# FILL THESE IN after creating KA, Genie, MAS:
MAS_ENDPOINT = "mas-XXXXXXXX-endpoint"  # From Step 5
KA_ENDPOINT = "ka-XXXXXXXX-endpoint"    # From Step 3
GENIE_SPACE_ID = "XXXXXXXX"             # From Step 4
LAKEBASE_INSTANCE = "kaapi-bricks"       # Created by setup script
SQL_WAREHOUSE_ID = "13a6b534c35ffbcb"    # Serverless Starter

print(f"""
=== APP.YAML CONFIG FOR INDIA WORKSPACE ===

command: ["python", "app.py"]

env:
  - name: MAS_ENDPOINT_NAME
    value: "{MAS_ENDPOINT}"
  - name: LAKEBASE_INSTANCE_NAME
    value: "{LAKEBASE_INSTANCE}"
  - name: LAKEBASE_DATABASE_NAME
    value: "databricks_postgres"

resources:
  - name: mas-endpoint
    serving_endpoint:
      name: "{MAS_ENDPOINT}"
      permission: CAN_QUERY
  - name: ka-endpoint
    serving_endpoint:
      name: "{KA_ENDPOINT}"
      permission: CAN_QUERY
  - name: genie-space
    genie_space:
      space_id: "{GENIE_SPACE_ID}"
      permission: CAN_EDIT
  - name: embedding-endpoint
    serving_endpoint:
      name: "databricks-bge-large-en"
      permission: CAN_QUERY
  - name: lakebase
    database:
      instance_name: "{LAKEBASE_INSTANCE}"
      database_name: "databricks_postgres"
      permission: CAN_CONNECT_AND_CREATE
  - name: sql-warehouse
    sql_warehouse:
      id: "{SQL_WAREHOUSE_ID}"
      permission: CAN_USE
  - name: llama-endpoint
    serving_endpoint:
      name: "databricks-meta-llama-3-3-70b-instruct"
      permission: CAN_QUERY
  - name: claude-endpoint
    serving_endpoint:
      name: "databricks-claude-sonnet-4-6"
      permission: CAN_QUERY
""")

## Step 8: Deploy Apps

After filling in the config above, run these from your terminal:

```bash
# 1. Update app.yaml in kaapi-bricks-mcp-app with the config above
# 2. Also update CATALOG/SCHEMA in app.py

# Deploy main chat app
databricks sync ./kaapi-bricks-mcp-app /Workspace/Users/kunal.gaurav@databricks.com/kaapi-bricks-finale --profile fevm-india-gcc
databricks apps deploy kaapi-bricks-finale --source-code-path /Workspace/Users/kunal.gaurav@databricks.com/kaapi-bricks-finale --profile fevm-india-gcc

# Deploy MCP server app
databricks apps create kaapi-ops-mcp --profile fevm-india-gcc
databricks sync ./kaapi-ops-mcp /Workspace/Users/kunal.gaurav@databricks.com/kaapi-ops-mcp --profile fevm-india-gcc
databricks apps deploy kaapi-ops-mcp --source-code-path /Workspace/Users/kunal.gaurav@databricks.com/kaapi-ops-mcp --profile fevm-india-gcc
```